Povoamento de Dados- Tabela de Factos AtividadeCliente

Importação de Pacotes

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime

Extração das Informações Necessárias

In [9]:
cliente = pd.read_csv("../Dados Finais/dim_cliente.csv",  encoding="utf-8-sig")
instrutor = pd.read_csv("../Dados Finais/dim_instrutor.csv",  encoding="utf-8-sig")
campanha = pd.read_csv("../Dados Finais/dim_campanha.csv",  encoding="utf-8-sig")
treino = pd.read_csv("../Dados Finais/dim_treino.csv",  encoding="utf-8-sig")
aula = pd.read_csv("../Dados Finais/dim_aula.csv",  encoding="utf-8-sig")

idas_ginasio = pd.read_csv('../Fontes/idas_ginasio_set.csv', encoding='latin1')

idas_ginasio.head()

,data,id_cliente,id_treino,id_aula,id_campanha,hora_entrada,hora_saida,avaliacao_treino,avaliacao_aula,semana,dia_semana
0,2025-09-01,14,14.0,1.0,C004,18,20,5.0,3.0,2025-09-01,0
1,2025-09-01,78,20.0,1.0,C003,19,20,5.0,NaN,2025-09-01,0
2,2025-09-01,35,NaN,2.0,C007,19,22,NaN,4.0,2025-09-01,0
3,2025-09-01,59,16.0,1.0,NaN,15,17,NaN,1.0,2025-09-01,0
4,2025-09-01,52,12.0,NaN,NaN,15,18,1.0,NaN,2025-09-01,0


Verificação da Integridade dos Dados

In [10]:
idas_ginasio['id_treino'] = idas_ginasio['id_treino'].astype('Int64')
idas_ginasio['id_aula'] = idas_ginasio['id_aula'].astype('Int64')

#Verificação da integridade referencial para cada dimensão (analisar se os IDs batem certo)
clientes_faltam = set(idas_ginasio['id_cliente']) - set(cliente['cliente_id'])
treinos_faltam = set(idas_ginasio['id_treino'].dropna()) - set(treino['treino_id_atividade'])
aulas_faltam = set(idas_ginasio['id_aula'].dropna()) - set(aula['aula_id'])
campanhas_faltam = set(idas_ginasio['id_campanha'].dropna()) - set(campanha['campanha_id'])

clientes_faltam, treinos_faltam, aulas_faltam, campanhas_faltam


(set(), set(), set(), set())

Substituição dos IDs das dimensões pelas respetivas surrogate Keys

In [11]:
#Substituir IDs de negócio por surrogate keys (SK)
# Cliente
fact = idas_ginasio.merge(cliente[['cliente_id', 'cliente_sk']], left_on='id_cliente', right_on='cliente_id', how='left')
fact = fact.drop(columns=['id_cliente', 'cliente_id'])
fact = fact.rename(columns={'cliente_sk': 'sk_cliente'})

# Treino
fact = fact.merge(treino[['treino_id_atividade', 'treino_sk']], left_on='id_treino', right_on='treino_id_atividade', how='left')
fact = fact.drop(columns=['id_treino', 'treino_id_atividade'])
fact = fact.rename(columns={'treino_sk': 'sk_treino'})

# Aula
fact = fact.merge(aula[['aula_id', 'aula_sk']], left_on='id_aula', right_on='aula_id', how='left')
fact = fact.drop(columns=['id_aula', 'aula_id'])
fact = fact.rename(columns={'aula_sk': 'sk_aula'})

# Campanha
fact = fact.merge(campanha[['campanha_id', 'campanha_sk']], left_on='id_campanha', right_on='campanha_id', how='left')
fact = fact.drop(columns=['id_campanha', 'campanha_id'])
fact = fact.rename(columns={'campanha_sk': 'sk_campanha'})

# Adicionar fact_id sequencial
fact = fact.reset_index(drop=True)
fact['fact_id'] = fact.index + 1

# Reordenar colunas: fact_id, surrogate keys, depois restantes atributos
col_order = ['fact_id', 'sk_cliente', 'sk_treino', 'sk_aula', 'sk_campanha',
             'data', 'hora_entrada', 'hora_saida', 'avaliacao_treino', 'avaliacao_aula']
fact = fact[col_order]

# Mostrar estrutura final
fact.head(3)


,fact_id,sk_cliente,sk_treino,sk_aula,sk_campanha,data,hora_entrada,hora_saida,avaliacao_treino,avaliacao_aula
0,1,14,4.0,1.0,4.0,2025-09-01,18,20,5.0,3.0
1,2,78,10.0,1.0,3.0,2025-09-01,19,20,5.0,NaN
2,3,35,NaN,2.0,7.0,2025-09-01,19,22,NaN,4.0


Adicionar Tempo Permanência

In [12]:
fact['tempo_permanencia'] = fact['hora_saida'] - fact['hora_entrada']

Adicionar Calorias Queimadas

In [13]:
# Mapear calorias do treino
calorias_treino = treino.set_index('treino_sk')['treino_calorias'].to_dict()

# Mapear calorias da aula
calorias_aula = aula.set_index('aula_sk')['aula_calorias'].to_dict()

def calcular_calorias(row):
    cal_treino = calorias_treino.get(row['sk_treino'], 0) if not pd.isna(row['sk_treino']) else 0
    cal_aula = calorias_aula.get(row['sk_aula'], 0) if not pd.isna(row['sk_aula']) else 0
    return cal_treino + cal_aula

fact['calorias_queimadas'] = fact.apply(calcular_calorias, axis=1)

# Mostrar amostra
fact.head(5)

,fact_id,sk_cliente,sk_treino,sk_aula,sk_campanha,data,hora_entrada,hora_saida,avaliacao_treino,avaliacao_aula,tempo_permanencia,calorias_queimadas
0,1,14,4.0,1.0,4.0,2025-09-01,18,20,5.0,3.0,2,832
1,2,78,10.0,1.0,3.0,2025-09-01,19,20,5.0,NaN,1,850
2,3,35,NaN,2.0,7.0,2025-09-01,19,22,NaN,4.0,3,220
3,4,59,6.0,1.0,NaN,2025-09-01,15,17,NaN,1.0,2,903
4,5,52,2.0,NaN,NaN,2025-09-01,15,18,1.0,NaN,3,595


Exportação dos Dados

In [14]:
fact.to_csv("../Dados Finais/tf_cliente.csv", index=False, encoding="utf-8-sig")
print("Tabela de Factos Atividade Cliente pronta para carga!")
display(fact.head())

Tabela de Factos Atividade Cliente pronta para carga!


,fact_id,sk_cliente,sk_treino,sk_aula,sk_campanha,data,hora_entrada,hora_saida,avaliacao_treino,avaliacao_aula,tempo_permanencia,calorias_queimadas
0,1,14,4.0,1.0,4.0,2025-09-01,18,20,5.0,3.0,2,832
1,2,78,10.0,1.0,3.0,2025-09-01,19,20,5.0,NaN,1,850
2,3,35,NaN,2.0,7.0,2025-09-01,19,22,NaN,4.0,3,220
3,4,59,6.0,1.0,NaN,2025-09-01,15,17,NaN,1.0,2,903
4,5,52,2.0,NaN,NaN,2025-09-01,15,18,1.0,NaN,3,595
